# 小さい言語モデル：文章から数値、確率、生成へ

11:45から12分。朝の初読後に使います。対象は固定したQwen3-0.6B。ColabはT4（16 GB）、MacはMPSを**自分で指定**します。初回ダウンロード時間は演習時間に含めません。
API料金は発生しません。Colabの利用料金・割当ては別です。既定ではモデル取得・実行・保存を行いません。

In [ ]:
from pathlib import Path
import sys
PREPARE_COLAB = False  # 新しいColabで固定版を取得する時だけTrue
RELEASE_COMMIT = "81f2fee040d3cabcea64212d8978e6dc9b84573d"
RELEASE_SHA256 = "b6026b9f1149be6ad828f0d4230bd15409d358864c12f6e03e505da2f4ff0d04"
BOOTSTRAP_SHA256 = "921585cfa741df6f607960a4f520c9ee4a551266a113ee1aa0ad102160cc43bc"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, install_api=False)
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. モデル用の2ライブラリだけを準備する

In [ ]:
from seminar_lab.open_weight import configure_cache, MODEL_ID, REVISION
CACHE = configure_cache(ROOT)
INSTALL = False
if INSTALL:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--cache-dir', str(CACHE / 'pip'),
                    '-r', str(ROOT / 'requirements-open-weight.txt')], check=True)
print(MODEL_ID, REVISION)

## 2. 計算機と固定モデルを選ぶ
GPU未割当てはエラーです。CPUや別モデルに自動変更しません。

In [ ]:
DEVICE = 'cuda'  # Colab: cuda / Mac: mps / CPU: cpu
ALLOW_DOWNLOAD = False
LOAD_MODEL = False
if LOAD_MODEL:
    import torch
    from seminar_lab.open_weight import load_model
    tokenizer, model = load_model(DEVICE, allow_download=ALLOW_DOWNLOAD)
    print('device:', model.device, 'dtype:', model.dtype)
    print('実際のパラメータ数:', sum(p.numel() for p in model.parameters()))

## 3. 文をトークンに分ける

「京都の名物は」は仕組みを見る人工例です。日本語の文字とトークンが1対1かを確認します。1トークンだけ復号すると文字の一部分になる場合もあります。

In [ ]:
OBSERVE = False
TEXT = '京都の名物は'
if OBSERVE:
    ids = tokenizer.encode(TEXT, add_special_tokens=False)
    if not 0 < len(ids) <= 64: raise ValueError('観察は1〜64トークンの短文にしてください。')
    print(ids)
    print(tokenizer.convert_ids_to_tokens(ids))
    print('元の文へ戻す:', tokenizer.decode(ids))

## 4. トークンIDを数値の列へ変換する
各行が1トークン。先頭4要素から意味を読み取れるとは限りません。文脈処理前の埋込みです。

In [ ]:
if OBSERVE:
    with torch.inference_mode():
        vectors = model.get_input_embeddings()(torch.tensor(ids, device=model.device))
    print('形（トークン数, 次元）:', tuple(vectors.shape))
    print(vectors[:, :4].float().cpu())

## 5. 次のトークンの候補確率を見る
softmaxで確率に直し、上位5個を表示。これは事実が正しい確率ではありません。

In [ ]:
if OBSERVE:
    encoded = tokenizer(TEXT, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        logits = model(**encoded).logits[0, -1].float()
    probabilities = torch.softmax(logits, dim=-1)
    values, candidates = torch.topk(probabilities, 5)
    for value, token_id in zip(values.cpu(), candidates.cpu()):
        print(repr(tokenizer.decode([int(token_id)])), float(value))

## 6. 自分の質問と入力範囲を用意する

問いは空欄から書きます。下の`source_excerpt`を加える際は資料の利用条件を確認。入力は2,048トークンまで、超えた場合は停止します。Methodsは必要な段落を自分で選びます。
会話テンプレートは全文を表示します。モデル固有の制御記号を含み、非公開の思考過程を取得する機能ではありません。

In [ ]:
PROMPT = ''
source_excerpt = ''
source_location = ''
PREVIEW = False
if PREVIEW:
    from seminar_lab.open_weight import encode_prompt
    full_prompt = source_excerpt + '\n\n' + PROMPT if source_excerpt else PROMPT
    visible_template, encoded = encode_prompt(tokenizer, full_prompt)
    print(visible_template)
    print('入力トークン数:', encoded.input_ids.shape[1])

## 7. 短い回答を1回生成する

新規会話、seed 42、temperature 0.7、top_p 0.8、top_k 20、出力96トークン。関数内の`model.generate`にこれらを渡します。出力上限で止まった回答は未完了と表示し、そのまま残します。同じseedでも装置間の完全一致は保証しません。

In [ ]:
RUN = False
if RUN:
    from seminar_lab.open_weight import generate
    if not PROMPT.strip(): raise ValueError('質問を記入してください。')
    full_prompt = source_excerpt + '\n\n' + PROMPT if source_excerpt else PROMPT
    result = generate(tokenizer, model, full_prompt, max_new_tokens=96, seed=42)
    print(result['response_raw'])
    print('終了状態:', result['status'], '秒:', result['elapsed_seconds'])

## 8. 生回答を上書きせず保存する

仕組み観察と論文の結果を区別。実行番号ごとのJSONを保存します。同じ番号は上書きしません。論文比較には次のセルで共通形式も保存できます。

In [ ]:
SAVE = False
if SAVE:
    import json
    if "result" not in globals():
        raise RuntimeError("保存する生成結果がありません。")
    result["student_explanation"] = ""  # 自分が説明できたこと
    result["unresolved_point"] = ""  # 確認できていないこと
    folder = ROOT / "outputs/open-weight"
    if not folder.resolve().is_relative_to(ROOT):
        raise ValueError("保存先は教材フォルダ内にしてください。")
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / (result["run_id"] + ".json")
    with destination.open("x", encoding="utf-8") as handle:
        json.dump(result, handle, ensure_ascii=False, indent=2)
    print("保存:", destination)

## 9. 論文の比較としても保存する場合

`P2a`は共通3問、`METHODS`は追加5問、`TRANSFER`はThe AI Scientist。質問・抜粋・生回答・本人の説明を共通のJSONL/Markdown形式で記録します。

In [ ]:
SAVE_READING = False
activity_id = 'METHODS'
student_explanation = ''
evidence_location = ''
if SAVE_READING:
    from seminar_lab.config import load_yaml, material_context
    from seminar_lab.open_weight import reading_record
    from seminar_lab.ui import save_pair
    course = load_yaml(ROOT / 'config/course.yaml')
    manifest = load_yaml(ROOT / 'materials/manifest.yaml')
    context = material_context(course, manifest, activity_id, for_input=False)
    record = reading_record(result, context, source_location)
    record.update(student_explanation=student_explanation, evidence_location=evidence_location)
    print([p.name for p in save_pair([record], ROOT / 'outputs')])

## 参照

[Qwenのモデル説明](https://huggingface.co/Qwen/Qwen3-0.6B)、[Transformers](https://huggingface.co/docs/transformers/index)、[原著論文と講義の対応](../docs/lecture_references.md)。モデルの取得・型・長さ制限・記録処理は`src/seminar_lab/open_weight.py`。論文の研究コードやエージェントを実行する演習ではありません。